# Transformer Encoder 与 Decoder

上一篇实现了 Encoder，这一篇补全 Decoder，组成完整的 Transformer。

### Encoder vs Decoder 一句话区别

- **Encoder**：看整个输入，理解上下文（双向注意力）
- **Decoder**：只能看到当前及之前的输出，逐步生成（单向注意力）

### 应用场景

| 任务 | 用什么 |
|------|--------|
| 文本分类、特征提取 | 只用 Encoder（BERT） |
| 文本生成、翻译 | Encoder + Decoder（原始 Transformer） |
| 文本生成（自回归） | 只用 Decoder（GPT） |
| 图像分类 | 只用 Encoder（ViT） |

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"设备: {device}")

设备: cuda


## 1. Decoder 的三个关键区别

### 区别一：Masked Self-Attention（掩码自注意力）

Decoder 生成文本时是一个一个 token 生成的：

```
生成第 1 个 token：只能看 "<start>"
生成第 2 个 token：能看 "<start> A"
生成第 3 个 token：能看 "<start> A B"
```

如果不加掩码，生成第 1 个 token 时就能看到后面的答案，等于作弊。

用一个上三角掩码实现：把未来位置的注意力分数设为 -inf，Softmax 后变成 0。

### 区别二：Cross-Attention（交叉注意力）

Decoder 需要参考 Encoder 的输出（源语言的语义），这就是交叉注意力：

```
Q 来自 Decoder（我在找什么）
K, V 来自 Encoder（源语言提供什么）
```

### 区别三：两层注意力

每个 Decoder Block 有两层注意力：

```
1. Masked Self-Attention：看已生成的 token
2. Cross-Attention：看 Encoder 的输出
```

In [2]:
# 掩码演示
seq_len = 5

# 上三角掩码：对角线以上全是 True（被屏蔽）
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
print("掩码（True = 不能看）:")
print(mask.int())
print()
print("解释:")
print("位置0 只能看位置0")
print("位置1 能看位置0、1")
print("位置2 能看位置0、1、2")
print("...")

# 加掩码后的注意力分数
scores = torch.randn(1, seq_len, seq_len)
scores_masked = scores.masked_fill(mask, float('-inf'))
attn = F.softmax(scores_masked, dim=-1)

print(f"\n加掩码后 Softmax 输出:")
print(attn[0].detach().numpy().round(3))
print("每行加起来=1，但看不到未来位置")

掩码（True = 不能看）:
tensor([[0, 1, 1, 1, 1],
        [0, 0, 1, 1, 1],
        [0, 0, 0, 1, 1],
        [0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0]], dtype=torch.int32)

解释:
位置0 只能看位置0
位置1 能看位置0、1
位置2 能看位置0、1、2
...

加掩码后 Softmax 输出:
[[1.    0.    0.    0.    0.   ]
 [0.026 0.974 0.    0.    0.   ]
 [0.067 0.301 0.632 0.    0.   ]
 [0.336 0.272 0.345 0.047 0.   ]
 [0.084 0.035 0.47  0.063 0.349]]
每行加起来=1，但看不到未来位置


## 2. Cross-Attention 演示

Cross-Attention 和 Self-Attention 的唯一区别：K 和 V 来自另一个序列。

In [3]:
def cross_attention(Q, K, V, d_k):
    """
    Q: 来自 Decoder (batch, tgt_len, d_model)
    K, V: 来自 Encoder (batch, src_len, d_model)
    """
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

# 模拟：翻译 "猫坐在垫子上" → "cat sat on mat"
encoder_out = torch.randn(1, 5, 64)  # 源语言：5个token
decoder_out = torch.randn(1, 4, 64)  # 目标语言：4个token

out, weights = cross_attention(decoder_out, encoder_out, encoder_out, d_k=64)
print(f"Encoder 输出: {encoder_out.shape}  (源语言 5 个 token)")
print(f"Decoder 输出: {decoder_out.shape}  (目标语言 4 个 token)")
print(f"Cross-Attention 输出: {out.shape}")
print(f"注意力权重: {weights.shape}  (4×5，每个目标token关注5个源token)")
print(f"\n权重（每行=一个目标token对源token的关注度）:")
print(weights[0].detach().numpy().round(3))

Encoder 输出: torch.Size([1, 5, 64])  (源语言 5 个 token)
Decoder 输出: torch.Size([1, 4, 64])  (目标语言 4 个 token)
Cross-Attention 输出: torch.Size([1, 4, 64])
注意力权重: torch.Size([1, 4, 5])  (4×5，每个目标token关注5个源token)

权重（每行=一个目标token对源token的关注度）:
[[0.195 0.319 0.067 0.097 0.323]
 [0.043 0.38  0.484 0.021 0.071]
 [0.247 0.133 0.141 0.328 0.152]
 [0.277 0.042 0.314 0.03  0.336]]


## 3. Decoder Block

一个 Decoder Block 的结构：

```
x → Masked Self-Attention → Add & Norm
  → Cross-Attention (Q=x, KV=encoder_out) → Add & Norm
  → FFN → Add & Norm → 输出
```

比 Encoder 多了一层 Cross-Attention。

In [4]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.d_k = d_model // num_heads
        self.num_heads = num_heads

        # 第一层：Masked Self-Attention
        self.W_q1 = nn.Linear(d_model, d_model)
        self.W_k1 = nn.Linear(d_model, d_model)
        self.W_v1 = nn.Linear(d_model, d_model)
        self.W_o1 = nn.Linear(d_model, d_model)
        self.norm1 = nn.LayerNorm(d_model)

        # 第二层：Cross-Attention
        self.W_q2 = nn.Linear(d_model, d_model)
        self.W_k2 = nn.Linear(d_model, d_model)
        self.W_v2 = nn.Linear(d_model, d_model)
        self.W_o2 = nn.Linear(d_model, d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # 第三层：FFN
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def _attention(self, Q, K, V, W_q, W_k, W_v, W_o, mask=None):
        batch_size = Q.size(0)
        Q = W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return W_o(out)

    def forward(self, x, encoder_out):
        # 生成掩码
        seq_len = x.size(1)
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()

        # 1. Masked Self-Attention
        attn1 = self._attention(x, x, x, self.W_q1, self.W_k1, self.W_v1, self.W_o1, mask)
        x = self.norm1(x + self.dropout(attn1))

        # 2. Cross-Attention
        attn2 = self._attention(x, encoder_out, encoder_out, self.W_q2, self.W_k2, self.W_v2, self.W_o2)
        x = self.norm2(x + self.dropout(attn2))

        # 3. FFN
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_out))

        return x

# 测试
decoder_block = DecoderBlock(d_model=64, num_heads=8, d_ff=256).to(device)
encoder_out = torch.randn(2, 10, 64).to(device)  # Encoder 输出
decoder_in = torch.randn(2, 8, 64).to(device)    # Decoder 输入

out = decoder_block(decoder_in, encoder_out)
print(f"Encoder 输出: {encoder_out.shape}")
print(f"Decoder 输入: {decoder_in.shape}")
print(f"Decoder 输出: {out.shape}")
print(f"参数量: {sum(p.numel() for p in decoder_block.parameters()):,}")

Encoder 输出: torch.Size([2, 10, 64])
Decoder 输入: torch.Size([2, 8, 64])
Decoder 输出: torch.Size([2, 8, 64])
参数量: 66,752


## 4. 完整 Transformer

把 Encoder 和 Decoder 组合在一起：

```
源语言 → Embedding + PosEnc → N × Encoder Block → encoder_out
                                                      ↓
目标语言 → Embedding + PosEnc → N × Decoder Block ──→ 输出 → Linear → Softmax
```

In [6]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.d_k = d_model // num_heads
        self.num_heads = num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.norm1 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size = x.size(0)
        Q = self.W_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(attn, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        x = self.norm1(x + self.dropout(self.W_o(attn_out)))

        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, max_len=512):
        super().__init__()
        # Encoder
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])

        # Decoder
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_dec = PositionalEncoding(d_model, max_len)
        self.decoder_layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])

        # 输出层
        self.output_linear = nn.Linear(d_model, tgt_vocab_size)

    def encode(self, src):
        x = self.pos_enc(self.src_embedding(src))
        for layer in self.encoder_layers:
            x = layer(x)
        return x

    def decode(self, tgt, encoder_out):
        x = self.pos_dec(self.tgt_embedding(tgt))
        for layer in self.decoder_layers:
            x = layer(x, encoder_out)
        return self.output_linear(x)

    def forward(self, src, tgt):
        encoder_out = self.encode(src)
        output = self.decode(tgt, encoder_out)
        return output

# 测试：模拟翻译任务
model = Transformer(
    src_vocab_size=1000,   # 源语言词表
    tgt_vocab_size=800,    # 目标语言词表
    d_model=64,
    num_heads=8,
    d_ff=256,
    num_layers=4,
).to(device)

src = torch.randint(0, 1000, (2, 15)).to(device)  # 源：15个token
tgt = torch.randint(0, 800, (2, 10)).to(device)   # 目标：10个token

output = model(src, tgt)
print(f"源语言: {src.shape}")
print(f"目标语言: {tgt.shape}")
print(f"输出: {output.shape}  (batch, tgt_len, tgt_vocab_size)")
print(f"\n参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"\n输出的最后一个维度是词表大小，每个位置预测下一个 token 的概率")

源语言: torch.Size([2, 15])
目标语言: torch.Size([2, 10])
输出: torch.Size([2, 10, 800])  (batch, tgt_len, tgt_vocab_size)

参数量: 634,144

输出的最后一个维度是词表大小，每个位置预测下一个 token 的概率


## 5. 训练 vs 推理

### 训练时（Teacher Forcing）

把整个目标序列一次性输入 Decoder，用掩码保证不看未来：

```
输入: <start> 猫 坐 在
标签: 猫   坐  在 垫子
```

### 推理时（自回归生成）

一个一个 token 生成：

```
第1步: <start> → 预测 "猫"
第2步: <start> 猫 → 预测 "坐"
第3步: <start> 猫 坐 → 预测 "在"
第4步: <start> 猫 坐 在 → 预测 "垫子"
第5步: <start> 猫 坐 在 垫子 → 预测 <end>
```

In [7]:
# 推理时的自回归生成演示
model.eval()

src = torch.randint(0, 1000, (1, 10)).to(device)  # 一句源语言
encoder_out = model.encode(src)

# 从 <start>=0 开始生成
generated = [0]  # 起始 token
max_len = 15

for _ in range(max_len):
    tgt = torch.tensor([generated]).to(device)
    output = model.decode(tgt, encoder_out)
    next_token = output[0, -1].argmax().item()  # 取最后一个位置的预测
    generated.append(next_token)

    if next_token == 1:  # <end>=1
        break

print(f"生成的 token 序列: {generated}")
print(f"生成长度: {len(generated)}")
print(f"\n注意：每一步都要重新跑整个 Decoder，因为输入变长了")
print(f"这就是自回归生成慢的原因——无法并行")

生成的 token 序列: [0, 310, 274, 465, 162, 397, 465, 162, 774, 508, 7, 85, 280, 560, 367, 711]
生成长度: 16

注意：每一步都要重新跑整个 Decoder，因为输入变长了
这就是自回归生成慢的原因——无法并行


## 总结

### Encoder
- 双向注意力：每个 token 能看到所有 token
- 用途：理解输入（BERT、ViT）

### Decoder
- 单向注意力（Masked）：只能看到当前及之前的 token
- 交叉注意力（Cross-Attention）：参考 Encoder 输出
- 用途：生成输出（GPT、翻译）

### 完整 Transformer

```
Encoder: 源语言 → 理解上下文 → 语义表示
Decoder: 目标语言 + 语义表示 → 逐步生成翻译
```

### 现代变体

| 模型 | 架构 | 用途 |
|------|------|------|
| BERT | 只有 Encoder | 文本理解、分类 |
| GPT | 只有 Decoder | 文本生成 |
| 原始 Transformer | Encoder + Decoder | 翻译、摘要 |
| ViT | 只有 Encoder | 图像分类 |